# Stage 1 Manual Forward For One Input

This notebook manually recomputes the Stage 1 transformer output for one input sequence. It is self-contained for Google Colab or another machine: upload `model.pt`, and optionally `config.json` if the model architecture changed from the default Stage 1 config.

## 1. Imports And File Paths

In [ ]:
# Colab usually already includes torch. Uncomment only if torch is missing.
# %pip install torch pandas

import csv
import json
import math
from dataclasses import asdict, dataclass, fields
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn

try:
    import pandas as pd
except ImportError:
    pd = None

torch.set_grad_enabled(False)
print(f'torch={torch.__version__}, cuda_available={torch.cuda.is_available()}')

In [ ]:
# Upload model.pt in Colab. This block is safe to skip outside Colab.
try:
    from google.colab import files
    print('Colab detected. Upload model.pt, and optionally config.json.')
    uploaded = files.upload()
except Exception:
    uploaded = {}

MODEL_PATH = Path('model.pt')
CONFIG_PATH = Path('config.json') if Path('config.json').exists() else None
OUTPUT_DIR = Path('manual_forward_single_input')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not MODEL_PATH.exists():
    candidates = sorted(Path('.').glob('*.pt')) + sorted(Path('.').glob('*.pth'))
    raise FileNotFoundError(f'MODEL_PATH does not exist: {MODEL_PATH}. Candidates: {[str(path) for path in candidates]}')

print('MODEL_PATH =', MODEL_PATH)
print('CONFIG_PATH =', CONFIG_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('DEVICE =', DEVICE)

## 2. Choose One Input

Set `TOKENS` to a list of token ids if you want to analyze an exact sequence. If `TOKENS = None`, the notebook generates one controlled sequence.

In [ ]:
# Example direct input: TOKENS = [2, 5, 1, 7, 8, 3, 4, 6, 9, 10]
TOKENS = None

# Used only when TOKENS is None.
LENGTH = 10
LABEL_TYPE = 'positive'  # 'positive' or 'negative'
TARGET_INDEX = None  # None means middle position for positive examples.
SEED = 12345

# Query positions to inspect in detail. The automatic aliases are useful for long sequences.
QUERY_ALIASES = ['target', 'last', 'max_target_attention']
TOP_K_KEYS = 8

## 3. Self-Contained Model Code

In [ ]:
@dataclass(frozen=True)
class TaskConfig:
    vocab_size: int = 16
    target_token: int = 1
    eval_lengths: tuple[int, ...] = (10, 20, 50, 100, 200, 500, 700, 900, 1000)
    positive_fraction: float = 0.5

    def __post_init__(self):
        object.__setattr__(self, 'eval_lengths', tuple(self.eval_lengths))

    def to_dict(self):
        data = asdict(self)
        data['eval_lengths'] = list(self.eval_lengths)
        return data


@dataclass(frozen=True)
class Stage1Config:
    seed: int = 1234
    device: str = 'auto'
    train_length: int = 10
    train_examples: int = 50000
    val_examples: int = 10000
    test_examples: int = 10000
    diagnostic_examples: int = 2000
    batch_size: int = 512
    eval_batch_size: int = 32
    epochs: int = 10
    learning_rate: float = 1e-3
    weight_decay: float = 0.0
    d_model: int = 64
    num_heads: int = 1
    num_layers: int = 1
    dim_feedforward: int = 128
    dropout: float = 0.0
    output_dir: str = 'runs/stage1_transformer_maxpool'

    def to_dict(self):
        return asdict(self)


class AttentionExportEncoderLayer(nn.Module):
    def __init__(self, *, d_model, num_heads, dim_feedforward, dropout):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.ReLU()

    def forward(self, hidden, *, return_attention):
        attention_output, attention_weights = self.self_attn(
            hidden,
            hidden,
            hidden,
            need_weights=return_attention,
            average_attn_weights=False,
        )
        hidden = self.norm1(hidden + self.dropout1(attention_output))
        feedforward_output = self.linear2(self.dropout(self.activation(self.linear1(hidden))))
        hidden = self.norm2(hidden + self.dropout2(feedforward_output))
        return hidden, attention_weights


class TransformerTokenPresenceClassifier(nn.Module):
    def __init__(self, *, vocab_size, d_model, num_heads, num_layers, dim_feedforward, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            AttentionExportEncoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(d_model, 1)

    def encode(self, tokens, *, return_attention):
        hidden = self.embedding(tokens)
        attention_weights = []
        for layer in self.layers:
            hidden, weights = layer(hidden, return_attention=return_attention)
            if weights is not None:
                attention_weights.append(weights)
        return hidden, attention_weights

    def forward(self, tokens):
        encoded, _ = self.encode(tokens, return_attention=False)
        pooled = encoded.max(dim=1).values
        return self.classifier(pooled).squeeze(-1)


def dataclass_values_from_metadata(config_class, values):
    valid_names = {field.name for field in fields(config_class)}
    return {key: value for key, value in values.items() if key in valid_names}


def load_model(model_path, config_path, device):
    if config_path is not None and Path(config_path).exists():
        metadata = json.loads(Path(config_path).read_text(encoding='utf-8'))
        task = TaskConfig(**dataclass_values_from_metadata(TaskConfig, metadata.get('task', {})))
        config = Stage1Config(**dataclass_values_from_metadata(Stage1Config, metadata.get('stage1', {})))
    else:
        metadata = {'note': 'default Stage 1 config used because config.json was not provided'}
        task = TaskConfig()
        config = Stage1Config()

    model = TransformerTokenPresenceClassifier(
        vocab_size=task.vocab_size,
        d_model=config.d_model,
        num_heads=config.num_heads,
        num_layers=config.num_layers,
        dim_feedforward=config.dim_feedforward,
        dropout=config.dropout,
    ).to(device)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, task, config, metadata


model, task, config, metadata = load_model(MODEL_PATH, CONFIG_PATH, DEVICE)
print('Loaded model')
print('task =', task.to_dict())
print('stage1 =', config.to_dict())

## 4. Build The Input Sequence

In [ ]:
def sample_non_target_tokens(length, *, task, generator):
    choices = torch.tensor([token for token in range(task.vocab_size) if token != task.target_token], dtype=torch.long)
    indices = torch.randint(len(choices), (length,), generator=generator)
    return choices[indices]


def make_controlled_tokens(*, length, label_type, target_index, task, seed):
    generator = torch.Generator().manual_seed(seed)
    tokens = sample_non_target_tokens(length, task=task, generator=generator)
    if label_type == 'positive':
        index = length // 2 if target_index is None else int(target_index)
        if index < 0 or index >= length:
            raise ValueError('target_index is out of range')
        tokens[index] = task.target_token
        return tokens, index
    if label_type == 'negative':
        return tokens, None
    raise ValueError("LABEL_TYPE must be 'positive' or 'negative'")


if TOKENS is None:
    tokens, target_index = make_controlled_tokens(
        length=LENGTH,
        label_type=LABEL_TYPE,
        target_index=TARGET_INDEX,
        task=task,
        seed=SEED,
    )
else:
    tokens = torch.tensor(TOKENS, dtype=torch.long)
    target_positions = torch.nonzero(tokens.eq(task.target_token), as_tuple=False).flatten().tolist()
    target_index = target_positions[0] if target_positions else None

if tokens.min().item() < 0 or tokens.max().item() >= task.vocab_size:
    raise ValueError('token ids must be within the configured vocabulary range')

print('length =', len(tokens))
print('target_index =', target_index)
print('tokens =', tokens.tolist())

## 5. Manual Forward Pass

In [ ]:
def split_qkv(layer):
    weight = layer.self_attn.in_proj_weight.detach()
    bias = layer.self_attn.in_proj_bias.detach()
    w_q, w_k, w_v = weight.chunk(3, dim=0)
    b_q, b_k, b_v = bias.chunk(3, dim=0)
    return {'w_q': w_q, 'w_k': w_k, 'w_v': w_v, 'b_q': b_q, 'b_k': b_k, 'b_v': b_v}


def manual_forward(model, tokens):
    if len(model.layers) != 1:
        raise ValueError('This notebook currently expects exactly one transformer layer.')
    device = next(model.parameters()).device
    tokens = tokens.to(device)
    layer = model.layers[0]
    hidden0 = model.embedding(tokens.unsqueeze(0)).squeeze(0)
    qkv = split_qkv(layer)

    q_full = F.linear(hidden0, qkv['w_q'], qkv['b_q'])
    k_full = F.linear(hidden0, qkv['w_k'], qkv['b_k'])
    v_full = F.linear(hidden0, qkv['w_v'], qkv['b_v'])

    length = hidden0.shape[0]
    num_heads = layer.self_attn.num_heads
    d_head = layer.self_attn.head_dim
    q = q_full.view(length, num_heads, d_head).transpose(0, 1)
    k = k_full.view(length, num_heads, d_head).transpose(0, 1)
    v = v_full.view(length, num_heads, d_head).transpose(0, 1)

    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_head)
    exp_scores = torch.exp(scores)
    denominators = exp_scores.sum(dim=-1)
    attention = exp_scores / denominators.unsqueeze(-1)
    attention_pre_out = torch.matmul(attention, v).transpose(0, 1).reshape(length, -1)
    attention_out = layer.self_attn.out_proj(attention_pre_out)
    after_attention = layer.norm1(hidden0 + layer.dropout1(attention_out))
    ff_hidden = layer.activation(layer.linear1(after_attention))
    feedforward = layer.linear2(layer.dropout(ff_hidden))
    encoded = layer.norm2(after_attention + layer.dropout2(feedforward))
    pooled, argmax_positions = encoded.max(dim=0)
    classifier_weight = model.classifier.weight.detach().squeeze(0)
    classifier_bias = model.classifier.bias.detach().squeeze()
    contributions = classifier_weight * pooled
    manual_logit = contributions.sum() + classifier_bias
    model_logit = model(tokens.unsqueeze(0)).squeeze()

    return {
        'tokens': tokens,
        'hidden0': hidden0,
        'q': q,
        'k': k,
        'v': v,
        'scores': scores,
        'exp_scores': exp_scores,
        'denominators': denominators,
        'attention': attention,
        'attention_pre_out': attention_pre_out,
        'attention_out': attention_out,
        'after_attention': after_attention,
        'ff_hidden': ff_hidden,
        'feedforward': feedforward,
        'encoded': encoded,
        'pooled': pooled,
        'argmax_positions': argmax_positions,
        'classifier_weight': classifier_weight,
        'classifier_bias': classifier_bias,
        'contributions': contributions,
        'manual_logit': manual_logit,
        'model_logit': model_logit,
    }


cache = manual_forward(model, tokens)
manual_logit = float(cache['manual_logit'].detach().cpu().item())
model_logit = float(cache['model_logit'].detach().cpu().item())
probability = float(torch.sigmoid(cache['manual_logit']).detach().cpu().item())
print('manual_logit =', manual_logit)
print('model_logit  =', model_logit)
print('absolute_diff =', abs(manual_logit - model_logit))
print('probability =', probability)
print('prediction =', 'positive' if manual_logit >= 0.0 else 'negative')

## 6. Inspect Intermediate Values

In [ ]:
def write_csv(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    fieldnames = list(rows[0])
    for row in rows[1:]:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def show(rows):
    return pd.DataFrame(rows) if pd is not None else rows


def resolve_query_indices(cache, target_index, aliases):
    length = cache['tokens'].shape[0]
    resolved = []
    target_attention = cache['attention'][0, :, target_index] if target_index is not None else None
    for alias in aliases:
        if alias == 'target' and target_index is not None:
            resolved.append(('target', target_index))
        elif alias == 'last':
            resolved.append(('last', length - 1))
        elif alias == 'first':
            resolved.append(('first', 0))
        elif alias == 'max_target_attention' and target_attention is not None:
            resolved.append(('max_target_attention', int(torch.argmax(target_attention).item())))
        elif isinstance(alias, int):
            resolved.append((f'query_{alias}', alias))
    unique = []
    seen = set()
    for name, index in resolved:
        if 0 <= index < length and (name, index) not in seen:
            seen.add((name, index))
            unique.append((name, index))
    return unique


query_rows = []
top_key_rows = []
selected_queries = resolve_query_indices(cache, target_index, QUERY_ALIASES)
for query_name, query_index in selected_queries:
    for head in range(cache['attention'].shape[0]):
        scores = cache['scores'][head, query_index].detach().cpu()
        exp_scores = cache['exp_scores'][head, query_index].detach().cpu()
        weights = cache['attention'][head, query_index].detach().cpu()
        denominator = float(exp_scores.sum().item())
        target_weight = float(weights[target_index].item()) if target_index is not None else float('nan')
        target_score = float(scores[target_index].item()) if target_index is not None else float('nan')
        non_target_exp_sum = denominator - (float(exp_scores[target_index].item()) if target_index is not None else 0.0)
        query_rows.append({
            'query_name': query_name,
            'query_index': query_index,
            'query_token': int(cache['tokens'][query_index].detach().cpu().item()),
            'head': head,
            'target_score': target_score,
            'softmax_denominator': denominator,
            'non_target_exp_sum': non_target_exp_sum,
            'attention_to_target': target_weight,
            'attention_entropy': float((-(weights.clamp_min(1e-12) * weights.clamp_min(1e-12).log()).sum()).item()),
        })
        top_values, top_indices = torch.topk(weights, k=min(TOP_K_KEYS, len(weights)))
        for rank, (key_index, value) in enumerate(zip(top_indices.tolist(), top_values.tolist()), start=1):
            top_key_rows.append({
                'query_name': query_name,
                'query_index': query_index,
                'head': head,
                'rank': rank,
                'key_index': key_index,
                'key_token': int(cache['tokens'][key_index].detach().cpu().item()),
                'is_target_key': key_index == target_index,
                'score': float(scores[key_index].item()),
                'attention_weight': float(value),
            })

pool_rows = []
argmax_positions = cache['argmax_positions'].detach().cpu()
encoded = cache['encoded'].detach().cpu()
pooled = cache['pooled'].detach().cpu()
classifier_weight = cache['classifier_weight'].detach().cpu()
contributions = cache['contributions'].detach().cpu()
for dim in range(len(pooled)):
    source_position = int(argmax_positions[dim].item())
    pool_rows.append({
        'dimension': dim,
        'source_position': source_position,
        'source_token': int(cache['tokens'][source_position].detach().cpu().item()),
        'source_is_target': source_position == target_index,
        'pooled_value': float(pooled[dim].item()),
        'classifier_weight': float(classifier_weight[dim].item()),
        'logit_contribution': float(contributions[dim].item()),
        'source_encoded_value': float(encoded[source_position, dim].item()),
    })

pool_rows_sorted = sorted(pool_rows, key=lambda row: abs(row['logit_contribution']), reverse=True)
summary_rows = [{
    'length': int(cache['tokens'].shape[0]),
    'target_index': target_index,
    'manual_logit': manual_logit,
    'model_logit': model_logit,
    'absolute_diff': abs(manual_logit - model_logit),
    'probability': probability,
    'prediction': 'positive' if manual_logit >= 0.0 else 'negative',
    'classifier_bias': float(cache['classifier_bias'].detach().cpu().item()),
    'positive_contribution_sum': float(contributions[contributions > 0].sum().item()),
    'negative_contribution_sum': float(contributions[contributions < 0].sum().item()),
}]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_csv(OUTPUT_DIR / 'summary.csv', summary_rows)
write_csv(OUTPUT_DIR / 'selected_query_attention.csv', query_rows)
write_csv(OUTPUT_DIR / 'selected_query_top_keys.csv', top_key_rows)
write_csv(OUTPUT_DIR / 'maxpool_dimension_contributions.csv', pool_rows)
(OUTPUT_DIR / 'input_tokens.json').write_text(json.dumps({'tokens': cache['tokens'].detach().cpu().tolist(), 'target_index': target_index}, indent=2) + '\n', encoding='utf-8')

print('Saved CSV files to', OUTPUT_DIR)

In [ ]:
show(summary_rows)

In [ ]:
show(query_rows)

In [ ]:
show(top_key_rows)

## 7. Full Attention Matrix

This saves one full attention matrix CSV per head. Rows are query positions and columns are key positions.

In [ ]:
ATTENTION_MATRIX_DISPLAY_LIMIT = 80
attention = cache['attention'].detach().cpu()
matrix_paths = []

for head in range(attention.shape[0]):
    matrix = attention[head]
    rows = []
    for query_index in range(matrix.shape[0]):
        row = {
            'query_index': query_index,
            'query_token': int(cache['tokens'][query_index].detach().cpu().item()),
        }
        for key_index in range(matrix.shape[1]):
            row[f'key_{key_index}_token_{int(cache["tokens"][key_index].detach().cpu().item())}'] = float(matrix[query_index, key_index].item())
        rows.append(row)

    path = OUTPUT_DIR / f'attention_matrix_head_{head}.csv'
    write_csv(path, rows)
    matrix_paths.append(path)

print('Saved full attention matrix CSV files:')
for path in matrix_paths:
    print('-', path)

if attention.shape[1] <= ATTENTION_MATRIX_DISPLAY_LIMIT and pd is not None:
    head = 0
    matrix = attention[head]
    labels = [f'{index}:tok{int(token)}' for index, token in enumerate(cache['tokens'].detach().cpu().tolist())]
    display(pd.DataFrame(matrix.numpy(), index=labels, columns=labels))
else:
    print(f'Matrix display skipped because length={attention.shape[1]} or pandas is unavailable. Use the CSV files instead.')

In [ ]:
# The largest absolute max-pool contributions explain which dimensions most affect the final logit.
show(pool_rows_sorted[:20])

## 8. How To Read This

- `manual_logit` should match `model_logit`; this verifies that the manual computation reproduces the model.
- `selected_query_attention.csv` shows the attention denominator and target attention for the target, last, and max-target-attention queries.
- `selected_query_top_keys.csv` shows which key positions each selected query attends to most strongly.
- `maxpool_dimension_contributions.csv` shows which sequence position won each pooled dimension and how that dimension contributes to the final logit.